# BrieFYI 요약 모델 QLoRA 파인튜닝 — Qwen3-8B (Colab 무료 T4용)

로컬 GPU가 6GB VRAM이라 8B급 QLoRA 학습이 안 되기 때문에, 코랩 무료 T4(약 15GB VRAM)에서
돌리기 위한 노트북입니다. 설계 문서: `docs/lora-finetune-summarization-design.md`
(특히 1.1절 코랩 시간 예상, 6절 코드 구조).

## 실행 전 준비물 (로컬 PC에서)

1. **`finetune_bundle.zip`** — `finetune/scripts/make_colab_bundle.sh` (Windows는 `.ps1`)를 실행해서 만듭니다.
   `src/`, `configs/`, `scripts/check_env.py`, `requirements-colab.txt`, `pyproject.toml`만 담겨 있습니다.
2. **`summarize_train.jsonl`, `summarize_val.jsonl`** — 로컬(DB 접속 가능한 환경)에서
   `python finetune/scripts/prepare_data.py --sources digest_pipeline`을 먼저 실행해 만든 결과물입니다.
   Colab에는 DB 접속 정보를 절대 올리지 않습니다 — 이미 정제된 JSONL 결과만 올립니다.
3. (선택) Hugging Face 계정 — Qwen3-8B는 공개 모델이라 보통 로그인 없이도 받아지지만,
   학습된 adapter를 나중에 Hub에 올리려면 로그인 토큰이 필요합니다.

## 세션이 끊겼을 때

체크포인트는 Google Drive(`MyDrive/briefyi-finetune/runs/...`)에 저장되므로, 코랩 세션이
끊기면 **이 노트북을 처음부터 다시 실행**하면 됩니다. `train.py`가 마지막 체크포인트를 자동으로
찾아 이어서 학습합니다 (`resume_from_checkpoint`).

## 1. GPU 확인

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 1))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

## 2. Google Drive 마운트

체크포인트를 여기에 저장합니다 — 세션이 끊겨도 안 날아갑니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/briefyi-finetune"
os.makedirs(f"{DRIVE_ROOT}/runs", exist_ok=True)
print("Drive 준비 완료:", DRIVE_ROOT)

## 3. `finetune_bundle.zip` 업로드

로컬에서 `finetune/scripts/make_colab_bundle.sh`(또는 `.ps1`)로 만든 zip 파일을 선택하세요.

In [ ]:
from google.colab import files
import zipfile, os

print("finetune_bundle.zip을 선택하세요")
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
os.makedirs("/content/finetune", exist_ok=True)
with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content/finetune")

print("압축 해제 완료:")
print(os.listdir("/content/finetune"))

## 4. 학습 데이터 업로드

로컬에서 `python finetune/scripts/prepare_data.py --sources digest_pipeline`로 미리 뽑아둔
`summarize_train.jsonl`, `summarize_val.jsonl` 두 파일을 함께 선택하세요.

(아직 준비가 안 됐고 파이프라인만 빨리 확인해보고 싶다면, 이 셀은 건너뛰고
8번 셀에서 `configs/smoke.yaml`용 `data/sample/` 경로를 대신 써도 됩니다 —
번들 zip 안에는 `data/sample`이 들어있지 않으니, 스모크 테스트를 하려면
`finetune/data/sample/*.jsonl` 두 파일도 이 셀에서 같이 업로드하세요.)

In [ ]:
from google.colab import files
import shutil, os

print("summarize_train.jsonl, summarize_val.jsonl 파일을 선택하세요")
data_uploaded = files.upload()

os.makedirs("/content/finetune/data/processed", exist_ok=True)
for name in data_uploaded:
    shutil.move(name, f"/content/finetune/data/processed/{name}")

print(os.listdir("/content/finetune/data/processed"))

## 5. 의존성 설치

`requirements-colab.txt`만 씁니다 — torch는 Colab에 이미 CUDA에 맞게 깔려 있어 재설치하지 않습니다.

In [ ]:
%cd /content/finetune
!pip install -q -r requirements-colab.txt

In [ ]:
import sys
sys.path.insert(0, "/content/finetune/src")

import summarize_ft
print("summarize_ft import 성공:", summarize_ft.__version__)

## 6. 환경 점검

In [ ]:
!python scripts/check_env.py

## 7. Hugging Face 로그인

학습이 끝나면 병합된 모델을 바로 Hub로 push합니다(11절). 지금 로그인해두세요 —
토큰은 반드시 **Write** 권한으로 발급받아야 push가 됩니다 (huggingface.co/settings/tokens).

In [ ]:
# 방법 1(권장): 왼쪽 사이드바 열쇠 아이콘(Secrets)에 이름 HF_TOKEN으로 토큰을 등록해두면
# 세션이 끊겨도 다시 붙여넣을 필요 없이 자동으로 로그인됩니다. 노트북을 여러 번 이어
# 실행해야 하는 이 학습 특성상 이 방법이 더 편합니다.
# 방법 2: Secrets를 안 쓰면 아래에서 notebook_login() 위젯이 뜨고, 토큰을 붙여넣으면 됩니다
# (이 경우 토큰은 이 세션에서만 유지되고 세션이 끝나면 다시 입력해야 합니다).

from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
    login(token=token)
    print("Colab Secrets(HF_TOKEN)로 로그인 완료")
except Exception:
    from huggingface_hub import notebook_login
    notebook_login()

## 8. 학습 설정 구성

`configs/qlora_qwen3-8b.yaml`을 베이스로, `output_dir`은 Drive 경로로, 데이터 경로는
방금 업로드한 파일로 덮어씁니다 (`--set`과 동일한 오버라이드 메커니즘, `config.py` 6.3절 참고).

스모크 테스트만 하고 싶다면 `overrides`의 데이터 경로를 `data/sample/summarize_*.sample.jsonl`로 바꾸세요.

In [ ]:
from summarize_ft.config import load_config, apply_overrides

RUN_NAME = "qwen3-8b-summarize-v1"
DRIVE_RUN_DIR = f"{DRIVE_ROOT}/runs/{RUN_NAME}"

cfg = load_config("/content/finetune/configs/qlora_qwen3-8b.yaml")
cfg = apply_overrides(cfg, [
    f"output_dir={DRIVE_RUN_DIR}",
    "data.train_path=/content/finetune/data/processed/summarize_train.jsonl",
    "data.val_path=/content/finetune/data/processed/summarize_val.jsonl",
    # Drive에는 재개(resume)에 필요한 최신 체크포인트 1개만 남긴다 — adapter+optimizer
    # 합쳐 ~150~200MB 수준으로 유지. 최종 결과물은 Drive가 아니라 뒤에서 바로 HF Hub로 push한다.
    "train.save_total_limit=1",
])

print("base_model:", cfg.base_model)
print("output_dir (Drive, 재개용 체크포인트만):", cfg.output_dir)
print("train_path:", cfg.data.train_path)
print("val_path:", cfg.data.val_path)
print("save_total_limit:", cfg.train.save_total_limit)

## 9. 학습 실행

1.1절 예상: Qwen3-8B는 스텝당 4~6초, 1 epoch(3,000건 기준) 3.5~5시간, 2~3 epoch 총 7~15시간.
코랩 무료 세션은 최대 12시간·유휴 90분 컷이라, 한 세션에 안 끝나면 **이 셀만 다시 실행**하면
Drive에 저장된 마지막 체크포인트부터 이어서 학습합니다.

In [ ]:
from summarize_ft.train import run_train

output_dir = run_train(cfg)
print("체크포인트 저장 위치:", output_dir)

## 10. 빠른 추론 확인 (선택)

학습이 끝난 뒤(또는 중간 체크포인트로) 실제로 요약이 나오는지 눈으로 확인합니다.

In [ ]:
from summarize_ft.infer import load_adapter_for_inference, summarize_hf

model, tokenizer = load_adapter_for_inference(cfg, output_dir)

sample_articles = [
    {
        "title": "국내 AI 스타트업, 시리즈B 투자 유치",
        "description": "국내 AI 스타트업 A사가 시리즈B 투자로 300억원을 유치했다고 밝혔다. 자체 LLM 연구개발을 확대할 계획이다.",
        "url": "https://example.com/news/1",
    }
]

result = summarize_hf(sample_articles, model, tokenizer)
print(result)

## 11. 병합 + Hugging Face Hub 업로드 (Drive에는 저장하지 않음)

Drive에는 재개용 체크포인트(어댑터+옵티마이저, ~150~200MB)만 남아 있습니다. 최종 결과물인
**병합된 전체 모델**(~16GB)은 Drive에 절대 쓰지 않고, `/content`(코랩 세션 임시 저장소)에서만
병합한 뒤 곧바로 Hugging Face Hub로 push하고 로컬 사본은 지웁니다.

병합은 메모리를 많이 씁니다. 아래 첫 셀에서 학습에 쓰던 GPU 메모리를 먼저 정리합니다.
그래도 메모리 에러가 나면 **런타임 재시작 후 이 11절부터** 다시 실행하세요 — 이때 8번 셀에서
`cfg`를 다시 만들고, 아래 두 번째 셀에서 `output_dir`을 `DRIVE_RUN_DIR`로 직접 지정하면 됩니다.

In [ ]:
import gc, torch

for name in ("trainer", "model", "tokenizer"):
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()
print("메모리 정리 완료")

In [ ]:
# 본인 HF 계정으로 바꾸세요. 이 이름이 로컬에서 SUMMARIZER_PROVIDER=hf로 호출할 때 쓰는 "모델명"이 됩니다.
HF_REPO_ID = "your-username/briefyi-qwen3-8b-summarize"

# output_dir이 이미 있으면(9번 셀을 이 세션에서 실행했으면) 그대로 쓰고,
# 런타임을 재시작했다면 아래 주석을 풀어 Drive의 체크포인트를 직접 가리키세요.
# output_dir = DRIVE_RUN_DIR

In [ ]:
import sys
sys.path.insert(0, "/content/finetune/src")

from summarize_ft.merge_lora import merge_and_save

MERGED_DIR = "/content/merged_model"        # Drive가 아니라 /content(로컬 임시 저장소)
OFFLOAD_DIR = "/content/merge_offload"       # 메모리가 부족할 때만 accelerate가 사용

merge_and_save(cfg.base_model, output_dir, MERGED_DIR, offload_folder=OFFLOAD_DIR)

In [ ]:
from huggingface_hub import HfApi

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(repo_id=HF_REPO_ID, folder_path=MERGED_DIR, repo_type="model")

print(f"업로드 완료: https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# 업로드가 끝났으면 /content의 로컬 사본은 지워서 세션 디스크를 비웁니다 (Drive에는 애초에 안 썼습니다).
import shutil

for path in (MERGED_DIR, OFFLOAD_DIR):
    shutil.rmtree(path, ignore_errors=True)
print("로컬 병합 사본 정리 완료")

## 12. 로컬(BrieFYI)에서 모델명으로 바로 사용하기

이제 `HF_REPO_ID`가 완전한 단일 모델(병합됨)로 Hub에 올라가 있으므로, 로컬 BrieFYI 코드에서는
base 모델과 adapter를 따로 챙길 필요 없이 이 이름 하나로 Claude API처럼 호출할 수 있습니다.

로컬(코랩 아님) `.env`에 추가:
```
HF_API_TOKEN="발급받은 토큰 (Read 권한이면 충분)"
HF_MODEL_ID="your-username/briefyi-qwen3-8b-summarize"
SUMMARIZER_PROVIDER=hf
```

로컬에서 바로 테스트:
```python
from tools.hf_llm_client import call_llm

print(call_llm("당신은 요약가입니다.", "다음을 한 문장으로 요약해줘: ..."))
```

실제 파이프라인 연동은 `tools/summarize_hf.py`의 `summarize_articles_hf`가 `tools/summarize.py`와
동일한 시그니처라 `agents/registry.py`의 summarizer tools에 `"summarize_hf": summarize_articles_hf`로
등록하고 `SUMMARIZER_PROVIDER=hf` 환경변수만 켜면 됩니다.

주의: 무료 서버리스 Inference API는 인기 모델 위주로 항상 켜져 있고, 막 올린 커스텀 파인튜닝
모델은 첫 호출 시 콜드스타트가 걸리거나(수십 초~수 분) 아예 서빙되지 않을 수 있습니다. 안정적으로
쓰려면 Hugging Face **Inference Endpoints**(유료, 전용 GPU)를 만들고 `tools/hf_llm_client.py`의
`InferenceClient(model=...)`를 endpoint URL로 바꾸면 됩니다 — 인터페이스는 그대로입니다.